In [ ]:
import json
import numpy as np
from pathlib import Path
import topological_insulator
from topological_insulator import Problem


# Locate repo root and structures dir from installed package

pkg_root = Path(topological_insulator.__file__).resolve().parent      # .../src/topological_insulator
repo_root = pkg_root.parent.parent                                    # .../topological_insulator-main
structures_dir = repo_root / "data" / "structures"

print("Repo root:", repo_root)
print("Structures dir:", structures_dir)
print("Has honeycomb.json?", (structures_dir / "honeycomb.json").exists())

# Helper: evaluate string expressions in JSON

def eval_expr(x):
    """Evaluate a simple numeric expression stored as string, like '3/2' or 'pow(3,0.5)/2'."""
    if isinstance(x, (int, float)):
        return float(x)
    if isinstance(x, str):
        return float(eval(x, {"__builtins__": None, "pow": pow}, {}))
    raise ValueError(f"Cannot evaluate expression: {x}")

# Build an (nx, ny) supercell JSON next to the original file
# IMPORTANT: only scale lattice vectors; do NOT touch delta_vectors
# so the number of basis sites and eigenvalue labels stay consistent.

def build_supercell(struct_name_in, struct_name_out, nx=2, ny=2):
    """Build an (nx, ny) supercell structure JSON next to the original file.

    We only scale the lattice vectors; delta_vectors (basis) are unchanged.
    This keeps the same sublattice labels (A, B) so the library doesn't
    look for eigenvalues.C, eigenvalues.D, etc.
    """
    in_path = structures_dir / struct_name_in
    out_path = structures_dir / struct_name_out

    with open(in_path, "r") as f:
        data = json.load(f)

    geom = data["geometry"]

    # Evaluate original lattice vectors
    lattice_raw = geom["lattice_vectors"]["value"]   # list of [expr_x, expr_y]
    lattice_vecs = np.array([[eval_expr(x) for x in vec] for vec in lattice_raw],
                            dtype=float)

    a1 = lattice_vecs[0]
    a2 = lattice_vecs[1]

    # New lattice vectors for nx x ny supercell
    new_a1 = (nx * a1).tolist()
    new_a2 = (ny * a2).tolist()

    # Create new data dict for supercell
    data_super = data.copy()
    data_super["general"]["name"]["value"] = f"{data['general']['name']['value']}_{nx}x{ny}"
    data_super["geometry"]["lattice_vectors"]["value"][0] = new_a1
    data_super["geometry"]["lattice_vectors"]["value"][1] = new_a2

    #  Deliberately do NOT modify geometry["delta_vectors"]["value"], eigenvalues, so the number of sublattices and eigenvalue labels (A, B) remain consistent.

    with open(out_path, "w") as f:
        json.dump(data_super, f, indent=2)

    print(f"Saved supercell structure to: {out_path}")
    print(f"Number of basis sites (unchanged): {len(data['geometry']['delta_vectors']['value'])}")
    return out_path

# Build a 2x2 supercell from honeycomb.json
build_supercell("honeycomb.json", "honeycomb_2x2.json", nx=2, ny=2)

# Set eigenvalue parameters (Kane–Mele style)

def set_eigenvalues(problem: Problem, t1, t2, M_val, B):
    """
    Set simple Kane–Mele-type parameters on the sublattices present
    in the geometry (A, B, ...).

    This follows the same access pattern as CellParser.set_eigenvalues:
      parser = getattr(self.eigenvalues, label_i).value
    """
    cell = problem.cell_parser

    # Magnetic field is a simple parameter with .value
    cell.field.magnetic.value = B

    # Use the same sublattice logic as the library:
    g = cell.geometry
    sublattice_labels = cell.sublattice_labels
    n_subs = len(g.delta_vectors.value)
    subs = sublattice_labels[:n_subs]   # e.g. ["A", "B"]

    for label_i in subs:
        # Each sublattice (A, B, ...) is a ParameterAttributes object
        ev_attr = getattr(cell.eigenvalues, label_i)  # e.g. cell.eigenvalues.A
        parser_i = ev_attr.value                      # underlying dict from JSON

        # Staggered mass term on s orbital
        M_here = M_val if label_i == "A" else -M_val
        if "onsite_energy" in parser_i and label_i in parser_i["onsite_energy"]:
            parser_i["onsite_energy"][label_i]["E_s"] = M_here

        # Nearest-neighbour hopping (t1)
        if "nn_hopping" in parser_i:
            for neighbour_label, hop_dict in parser_i["nn_hopping"].items():
                if "t_ss_sigma" in hop_dict:
                    hop_dict["t_ss_sigma"] = t1

        # Kane–Mele SOC on this sublattice
        # In honeycomb.json, this is keyed by the same label (A or B)
        if "kane_mele_soc" in parser_i and label_i in parser_i["kane_mele_soc"]:
            parser_i["kane_mele_soc"][label_i]["lambda_ss"] = t2

# Run 1x1 and 2x2 bulk problems

location = "bulk"
N_k = 81
N_r = 10   # must be >= 10 because of assert in Problem.setup

t1 = -1.0
t2 = 0.07 * t1
M_val = 0.0
B = 0.0

structure_path = str(structures_dir)

# 1x1 primitive bulk
problem_1x1 = Problem(structure_path=structure_path,
                      structure_name="honeycomb.json")

set_eigenvalues(problem_1x1, t1, t2, M_val, B)

problem_1x1.setup(
    N_r=N_r,
    N_k=N_k,
    location=location,
    BZ="reduced"
)
problem_1x1.run(H_type="reciprocal")

# 2x2 supercell bulk via lattice scaling only
problem_2x2 = Problem(structure_path=structure_path,
                      structure_name="honeycomb_2x2.json")

set_eigenvalues(problem_2x2, t1, t2, M_val, B)

problem_2x2.setup(
    N_r=N_r,    # same N_r for fair comparison
    N_k=N_k,
    location=location,
    BZ="reduced"
)
problem_2x2.run(H_type="reciprocal")

print("Done building bulk Hamiltonians for 1x1 and 2x2.")


Repo root: /Users/vedantdubey/Desktop/UCL/Year 4/Dissertation/topological_insulator-main
Structures dir: /Users/vedantdubey/Desktop/UCL/Year 4/Dissertation/topological_insulator-main/data/structures
Has honeycomb.json? True
Saved supercell structure to: /Users/vedantdubey/Desktop/UCL/Year 4/Dissertation/topological_insulator-main/data/structures/honeycomb_2x2.json
Number of basis sites (unchanged): 2
Building Geometry...
Geometry - Done.
Building 'Bulk' Hamiltonian...
'Bulk' Hamiltonian - Done.
Calculating 'Bulk' Eigenvalues...
'Bulk' Eigenvalues - Done!
Building Geometry...
Geometry - Done.
Building 'Bulk' Hamiltonian...
'Bulk' Hamiltonian - Done.
Calculating 'Bulk' Eigenvalues...
'Bulk' Eigenvalues - Done!
Done building bulk Hamiltonians for 1x1 and 2x2.


In [ ]:
import numpy as np

tb1 = problem_1x1.hamiltonian["bulk"]["tight_binding"]
tb2 = problem_2x2.hamiltonian["bulk"]["tight_binding"]

E1_dict = tb1.E_k_dict
E2_dict = tb2.E_k_dict

print("Number of k-points (1x1):", len(E1_dict))
print("Number of k-points (2x2):", len(E2_dict))

# Flatten all eigenvalues over all k-points for each system
E1_all = np.concatenate([np.ravel(v) for v in E1_dict.values()])
E2_all = np.concatenate([np.ravel(v) for v in E2_dict.values()])

print("Total eigenvalues (1x1):", E1_all.size)
print("Total eigenvalues (2x2):", E2_all.size)

# Sort the flattened spectra to compare as sets (ignoring which k they came from)

E1_sorted = np.sort(E1_all)
E2_sorted = np.sort(E2_all)

# If lengths differ, you already know they can't match

if E1_sorted.shape != E2_sorted.shape:
    print("Warning: different total number of eigenvalues, spectra cannot be identical.")
else:
    diff = np.abs(E1_sorted - E2_sorted)
    max_diff = np.max(diff)
    print("Max |E1 - E2| over full spectrum:", max_diff)

    tol = 1e-8
    print("Spectra identical within tol?", max_diff < tol)


Number of k-points (1x1): 7396
Number of k-points (2x2): 7396
Total eigenvalues (1x1): 118336
Total eigenvalues (2x2): 118336
Max |E1 - E2| over full spectrum: 1.9999999999999998
Spectra identical within tol? False
